# Seasonal Generator

Generate time series with configurable seasonal patterns, trend, and noise.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from synforecast.generators import SeasonalGenerator

## Define Generator Parameters

Configure a seasonal generator with daily seasonality (24-hour period), a slight upward trend, and a base level of 50.

In [ ]:
params = {
    "min_length": 168,
    "max_length": 336,
    "freq": "h",
    "seasonality_period": 24,
    "seasonality_amplitude": 15.0,
    "trend": 0.05,
    "noise_level": 2.0,
    "base_level": 50.0,
    "seed": 123,
}

generator = SeasonalGenerator(engine="polars", **params)
df = generator.generate(n_series=3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for uid in df["unique_id"].unique().to_list():
    series = df.filter(pl.col("unique_id") == uid)
    ax.plot(series["ds"].to_list(), series["y"].to_list(), label=uid, alpha=0.8)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Value")
ax.set_title("Seasonal Time Series")
ax.legend()
plt.tight_layout()
plt.show()

## Inspect the Generated Data

In [ ]:
print(f"Generated {df['unique_id'].n_unique()} time series")
print(f"Total observations: {len(df)}")
df.head(10)

## Statistics by Series

In [ ]:
df.group_by("unique_id").agg(
    [
        pl.col("y").count().alias("count"),
        pl.col("y").min().alias("min_value"),
        pl.col("y").max().alias("max_value"),
        pl.col("y").mean().alias("mean_value"),
        pl.col("y").std().alias("std_value"),
    ]
)

## Sample of One Series

View the first 24 hours of a single series to see the seasonal pattern.

In [ ]:
df.filter(pl.col("unique_id") == "0").head(24)